#Notebook used to clean recollected data
###The csv need to be uploaded into the content folder


Packages instalation

In [ ]:
pip install openai python-dotenv

####Imports needed to clean the texts

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
import openai
import time
from google.colab import files

###Load of the CSV and showing it for a better comparison.
###Is also determines where to start if there are some type of error cleaning the texts

In [ ]:
input_file = "EspeciesReadyToclean.csv"
output_path = "cleaned_result.csv"
df = pd.read_csv(input_file)
df

,Species,Text,Page_id,Volume,Year,Source,Source_ID,Institution,Language,Rights,Copyright
0,Transandinomys talamancae,AF251520 Hylaeamys yunganus AF251522 Hylaeamys...,64738101,6,2022,Virtual Item,vi210912v6no1202220250518010740,Pensoft Publishers,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
1,Transandinomys talamancae,Figure 3. Phylogenetic tree of maximum likelih...,64738101,6,2022,Virtual Item,vi210912v6no1202220250518010740,Pensoft Publishers,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
2,Transandinomys talamancae,Identification. Dorsum fur grayish with reddis...,64171121,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
3,Transandinomys talamancae,Identification. Dorsal fur short (8-10 mm) and...,64171121,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
4,Transandinomys talamancae,Identification. Length of head and body 276 mm...,64171121,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
...,...,...,...,...,...,...,...,...,...,...,...
3869,Cyathea wendlandii,The 1997 IUCN Red List of Threatened Plants Pt...,31085368,NaN,NaN,Internet Archive,1997iucnredlisto97walt,"UNEP-WCMC, Cambridge",English,NaN,Not provided. Contact Holding Institution to v...
3870,Cyathea wendlandii,Rut Cyathea urbanii Brause 12491 R= 12616 Domi...,31085368,NaN,NaN,Internet Archive,1997iucnredlisto97walt,"UNEP-WCMC, Cambridge",English,NaN,Not provided. Contact Holding Institution to v...
3871,Bunodophoron melanocarpum,Studies on North American Cortinarii. I. New a...,64545336,NaN,NaN,Internet Archive,mycotaxon55unse,Cornell University Library,English,http://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
3872,Bunodophoron melanocarpum,eae VAM kik ae ce ace ool ee wee We he ein ay ...,64545336,NaN,NaN,Internet Archive,mycotaxon55unse,Cornell University Library,English,http://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...


###Checking if the API KEY is well charged

In [ ]:
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("API Key cargada:", os.getenv("OPENAI_API_KEY"))

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

###The definition of all the functions used to cleaning the texts in the dataframe
####As it´s name says, the clean_text function is the one who uses the API KEY and asks the Api to clean the text
####The next one is the one who access the data in the dataframe and build the new csv, every 5 requests the function saves the csv in Colab and every 100 requests it downloads the actual cleaned csv



In [ ]:
def clean_text(text, language='english'):
    prompt = f"Clean and correct the following text in {language}:\n\n{text}"

    response = client.chat.completions.create(
        model="gpt-4",  # o "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You are a helpful assistant that improves the clarity and correctness of text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

def clean_dataframe(df, output_path):
    start_index = 0
    if os.path.exists(output_path):
        try:
            existing = pd.read_csv(output_path)
            start_index = len(existing[existing['Cleaned_Text'].notna()])
            df['Cleaned_Text'] = existing.get('Cleaned_Text', pd.NA)
        except:
            df['Cleaned_Text'] = pd.NA
    else:
        df['Cleaned_Text'] = pd.NA

    for i in range(start_index, len(df)):
        text = str(df.at[i, 'Text'])
        language = str(df.at[i, 'Language']).lower()
        if pd.isna(text) or text.strip() == "":
            continue
        print(f"Cleaning row {i} ({language})")
        try:
            cleaned = clean_text(text, language)
        except Exception as e:
            print(f"Error: {e}")
            cleaned = None
        df.at[i, 'Cleaned_Text'] = cleaned
        time.sleep(1.2)
        if (i + 1) % 5 == 0:
            df.to_csv(output_path, index=False)
            print(f"Saved progress at row {i}")
        if (i + 1) % 100 == 0:
            df.to_csv(output_path, index=False)
            print(f"Downloading backup at row {i}")
            files.download(output_path)

    df.to_csv(output_path, index=False)
    print("Done.")

###The call of the fuction that cleans the dataframe, since the function changes or creates the csv we only have to make it a dataframe to verify the new cleaned texts.

In [ ]:
clean_dataframe(df, output_path)
df = pd.read_csv("cleaned_result.csv")


Cleaning row 0 (spanish)
Ramírez-P, Castro-C, Arroyo-C, Cervantes 23. Tribu/Tribe Stenodermatini. 

Especies:

1. Artibeus hirsutus Andersen, 1906
2. Artibeus intermedius J. A. Allen, 1897
3. Artibeus intermedius intermedius J. A. Allen, 1897
4. Artibeus intermedius koopmani Wilson, 1991
5. Artibeus jamaicensis Leach, 1821
6. Artibeus jamaicensis planus Davis, 1970
7. Artibeus jamaicensis richardsoni J. A. Allen, 1908
8. Artibeus jamaicensis triomylus Handley, 1966
9. Artibeus jamaicensis yucatanicus J. A. Allen, 1904
10. Artibeus lituratus (Olfers, 1818)
11. Artibeus lituratus palmarum J. A. Allen y Chapman, 1897
12. Carollia brevicauda (Schinz, 1821)
13. Carollia perspicillata (Linnaeus, 1758)
14. Carollia perspicillata azteca Saussure, 1860
15. Carollia subrufa (Hahn, 1905)
16. Centurio senex Gray, 1842
17. Centurio senex senex Gray, 1842
18. Chiroderma salvini Dobson, 1878
19. Chiroderma salvini salvini Dobson, 1878
20. Chiroderma salvini scapaeum Handley, 1966
21. Chiroderma villo

,species,paragraph,page_id,Volumen,Año,Fuente,IdFuente,Institución,Idioma,Derechos,Copyright
0,Dermanura azteca,"Ramírez-P, Castro-C, Arroyo-C, Cervantes 23. T...",55106416,158,1996,Internet Archive,listataxonoymic158rami,Museum of Texas Tech University,Spanish,http://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
1,Dermanura azteca,LA BIODIVERSIDAD EN AGUASCALIENTES 166 \n\nCua...,50959835,NaN,2008,Internet Archive,biodiversidaden00Avil,Comisión Nacional para el Conocimiento y Uso d...,Spanish,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
2,Dermanura azteca,"González Christen, Especie Nt Ct Mo P1 Tu Pr A...",50680949,Appendix,2011,Internet Archive,LabiodiversidadAppeCruz,Comisión Nacional para el Conocimiento y Uso d...,Spanish,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
3,Dermanura azteca,"158. Biodiversidad en Chiapas, Apéndice VIII. ...",50746615,Apéndices (2013),2013,Internet Archive,biodiversidadenApndCruz,Comisión Nacional para el Conocimiento y Uso d...,Spanish,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
4,Dermanura azteca,Portada Índice Anterior Siguiente Créditos \n\...,50734895,Anexos (2005),2005,Internet Archive,LabiodiversidadAnexVill,Comisión Nacional para el Conocimiento y Uso d...,Spanish,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...


In [ ]:
print("The Original Text")
print(df['Text'].iloc[0])
print('/n')
print("The Cleaned Text")
print(df['Cleaned_Text'].iloc[0])

NameError: name 'df' is not defined